# AU alignment calibration

Calibrate `tau_mask`, `n_eff_min`, `q_c`, and `eta` using RAF-DB/ExpW train only. Do not use validation or test accuracy.

In [ ]:
from pathlib import Path
import torch
from cs_aualign_vit.data.datasets import build_loader

SOURCE_MANIFEST = Path("data/processed/raf-v5-deduplicated/processed.jsonl")
TARGET_MANIFEST = Path("/home/hung/FER-DomainAdaptation/CS-AUAlign-ViT/data/processed/expw-v1/processed.jsonl")
SOURCE_CHECKPOINT = Path("runs/source-pretraining-val/best.pt")
ADAPTATION_CHECKPOINT = Path("runs/adaptation/run-001/adaptation.pt")
SOURCE_AU_CACHE = Path("/path/to/raf_db_au.csv")
TARGET_AU_CACHE = Path("/path/to/expw_au.csv")
BATCH_SIZE = 64
SEED = 42
N_EFF_CANDIDATES = (2, 4, 8, 16)
R_TARGET = 0.1

In [ ]:
source_loader = build_loader(SOURCE_MANIFEST, "source_train", batch_size=BATCH_SIZE, seed=SEED, au_cache=SOURCE_AU_CACHE)
target_loader = build_loader(TARGET_MANIFEST, "target_adaptation", batch_size=BATCH_SIZE, seed=SEED, au_cache=TARGET_AU_CACHE)

def collect(loader):
    au, mass, au_valid, landmark_valid = [], [], [], []
    for _images, _identity, metadata in loader:
        au.append(metadata["au"])
        mass.append(metadata["mask"].sum(dim=1))
        au_valid.append(metadata["au_valid"])
        landmark_valid.append(metadata["landmark_valid"])
    return tuple(torch.cat(values) for values in (au, mass, au_valid, landmark_valid))

source = collect(source_loader)
target = collect(target_loader)
q_c = source[0].mean(dim=0) * target[0].mean(dim=0)
q_c

In [ ]:
def mask_report(values):
    au, mass, au_valid, landmark_valid = values
    quantiles = torch.tensor([0.01, 0.05, 0.10, 0.50, 0.95])
    return {
        "minimum": mass.amin(dim=0),
        "quantiles": torch.quantile(mass, quantiles, dim=0),
        "near_zero_rate": (mass <= 1e-8).float().mean(dim=0),
        "au_missing_rate": (~au_valid).float().mean(),
        "landmark_invalid_rate": (~landmark_valid).float().mean(),
    }

source_mask_report = mask_report(source)
target_mask_report = mask_report(target)
source_mask_report, target_mask_report

In [ ]:
# Set after inspecting both mask reports; do not select it from target accuracy.
TAU_MASK = None

def batch_coverage(loader, tau_mask, candidate):
    rows = []
    for _images, _identity, metadata in loader:
        mass = metadata["mask"].sum(dim=1)
        valid = metadata["au_valid"][:, None] & metadata["landmark_valid"][:, None] & (mass >= tau_mask)
        weights = metadata["au"] * valid
        sums = weights.sum(dim=0)
        n_eff = sums.square() / (weights.square().sum(dim=0) + 1e-8)
        rows.append((n_eff, (n_eff >= candidate) & (sums > 1e-8)))
    return torch.stack([row[0] for row in rows]), torch.stack([row[1] for row in rows])

# Run after setting TAU_MASK. Compare source and target coverage for every fixed candidate.
coverage = {} if TAU_MASK is None else {
    candidate: (batch_coverage(source_loader, TAU_MASK, candidate), batch_coverage(target_loader, TAU_MASK, candidate))
    for candidate in N_EFF_CANDIDATES
}
coverage

## Gradient calibration

Using a fixed list of train batches and the same encoder parameters, measure `||grad L_local|| / ||grad L_POST||`. For each batch compute `eta_0 = R_TARGET * ||grad L_POST|| / (||grad L_local|| + epsilon)`. Choose one fixed `eta` from the median and report P10/P90. Fail calibration when local gradients are almost always zero. Copy the selected values and supporting measurements to `reports/au_alignment_calibration.md`, then update `configs/training/adaptation.yaml`.

In [ ]:
def gradient_norm(loss, parameters):
    gradients = torch.autograd.grad(loss, parameters, retain_graph=True, allow_unused=True)
    squares = [gradient.detach().float().square().sum() for gradient in gradients if gradient is not None]
    return torch.stack(squares).sum().sqrt() if squares else torch.tensor(0.0, device=loss.device)

def eta_candidate(loss_post, loss_local, encoder_parameters, epsilon=1e-8):
    parameters = tuple(encoder_parameters)
    post_norm = gradient_norm(loss_post, parameters)
    local_norm = gradient_norm(loss_local, parameters)
    return {
        "post_grad_norm": post_norm,
        "local_grad_norm": local_norm,
        "raw_ratio": local_norm / (post_norm + epsilon),
        "eta_0": R_TARGET * post_norm / (local_norm + epsilon),
    }

# Call eta_candidate on each fixed pilot batch using the same encoder parameter tuple.